# Protocol 5 — fMRI Anticipation: Somatic Marker Retrieval vs. Prediction Error

Simulates predictions Pred 5.A–Pred 5.D from :

* **Prediction 5.A** — M̂ modulates Π²_eff, not ε² directly
* **Prediction 5.B** — vmPFC–posterior insula PPI peaks during anticipation, not outcome
* **Prediction 5.C** — vmPFC BOLD is sensitive to option valence (EV), not sensory contrast
* **Prediction 5.D** — Removing anticipation (no-foreperiod) collapses vmPFC–insula coupling

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel
from apgi.core import compute_pi_i_eff, compute_S_t, compute_theta_t, ignition_criterion

: 

## 1 — Protocol 5 parameters

In [ ]:
KAPPA = 100.0
ALPHA = 0.3
BETA = 0.7
GAMMA_V = 0.6
GAMMA_A = 0.3
PI_I_LONG = 1.2  # long foreperiod: somatic marker active
PI_I_SHORT = 0.8  # no foreperiod: somatic marker suppressed
N_SUBJECTS = 36

rng = np.random.default_rng(5)

## 2 — Simulate PPI coefficients by window (Prediction 5.B)

In [ ]:
# PPI anticipation > outcome (Pred 5.B)
ppi_anticipation = rng.normal(0.42, 0.12, N_SUBJECTS)
ppi_outcome = rng.normal(0.11, 0.10, N_SUBJECTS)

t_stat, p_val = ttest_rel(ppi_anticipation, ppi_outcome)
print(f"PPI anticipation: {ppi_anticipation.mean():.3f} ± {ppi_anticipation.std():.3f}")
print(f"PPI outcome:      {ppi_outcome.mean():.3f} ± {ppi_outcome.std():.3f}")
print(f"Paired t-test: t={t_stat:.3f}, p={p_val:.4f}")

## 3 — Prediction 5.C: vmPFC sensitive to valence, not contrast

In [ ]:
bold_valence = rng.normal(0.55, 0.14, N_SUBJECTS)
bold_contrast = rng.normal(0.08, 0.11, N_SUBJECTS)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Pred 5.B: PPI anticipation vs outcome
ax = axes[0]
ax.bar(
    ["Anticipation", "Outcome"],
    [ppi_anticipation.mean(), ppi_outcome.mean()],
    yerr=[
        ppi_anticipation.std() / np.sqrt(N_SUBJECTS),
        ppi_outcome.std() / np.sqrt(N_SUBJECTS),
    ],
    color=["#2166ac", "#9966FF"],
    alpha=0.85,
    edgecolor="white",
    width=0.4,
    capsize=5,
)
ax.axhline(0, ls="--", lw=0.8, color="k", alpha=0.4)
ax.set_title("Pred 5.B — PPI: Anticipation > Outcome")
ax.set_ylabel("vmPFC–pIC PPI coeff.")
ax.spines[["top", "right"]].set_visible(False)

# Pred 5.C: valence vs contrast sensitivity
ax = axes[1]
ax.bar(
    ["Option EV\n(valence)", "Sensory\ncontrast"],
    [bold_valence.mean(), bold_contrast.mean()],
    yerr=[
        bold_valence.std() / np.sqrt(N_SUBJECTS),
        bold_contrast.std() / np.sqrt(N_SUBJECTS),
    ],
    color=["#2166ac", "#d6604d"],
    alpha=0.85,
    edgecolor="white",
    width=0.4,
    capsize=5,
)
ax.axhline(0, ls="--", lw=0.8, color="k", alpha=0.4)
ax.set_title("Pred 5.C — vmPFC: Valence, not Contrast")
ax.set_ylabel("vmPFC BOLD β (a.u.)")
ax.spines[["top", "right"]].set_visible(False)

# Pred 5.D: foreperiod manipulation
ppi_long = rng.normal(0.44, 0.11, N_SUBJECTS)
ppi_none = rng.normal(0.09, 0.10, N_SUBJECTS)
ax = axes[2]
ax.bar(
    ["Long FP\n(2000–4000 ms)", "No FP\n(0 ms)"],
    [ppi_long.mean(), ppi_none.mean()],
    yerr=[ppi_long.std() / np.sqrt(N_SUBJECTS), ppi_none.std() / np.sqrt(N_SUBJECTS)],
    color=["#2166ac", "#AAAAAA"],
    alpha=0.85,
    edgecolor="white",
    width=0.4,
    capsize=5,
)
ax.axhline(0, ls="--", lw=0.8, color="k", alpha=0.4)
ax.set_title("Pred 5.D — Foreperiod Collapses Coupling")
ax.set_ylabel("vmPFC–pIC PPI coeff.")
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle("Protocol 5 — fMRI Anticipation (Pred 5.A–Pred 5.D)", fontsize=12)
plt.tight_layout()
plt.show()

## 4 — APGI interpretation

M̂(c,a) = γ_V · V(c,a) + γ_A · A(c,a) acts on Πₒ_eff *before* decision execution:

85735\Pi^i_{\text{eff, antici}} = \Pi^i_{\text{long FP}} \cdot e^{-C/\kappa}85735
85735\Pi^i_{\text{eff, no FP}} = \Pi^i_{\text{short}} \cdot e^{-C/\kappa}85735

The vmPFC signal in the anticipation window is the neural correlate of M̂ retrieval,
not outcome-locked εₒ. This dissociation is the core prediction of Pred 5.A.